# 6. Model card and summary

This notebook does not train anything. It reads the results saved by notebooks 4 and 5 (`results/*.json`) so every number here comes from code that ran, and puts the findings, the limits, and what I would do next in one place.

In [1]:
import json
from pathlib import Path

import pandas as pd

RESULTS = Path.cwd().parent / "results"
reg = json.loads((RESULTS / "regression.json").read_text())
clf = json.loads((RESULTS / "classification.json").read_text())
pd.set_option("display.precision", 3)

## Model 1: hourly bike rentals (regression)

| | |
|---|---|
| **Purpose** | Estimate rentals per hour given the calendar and that hour's weather. |
| **Data** | UCI Bike Sharing, Capital Bikeshare, Washington DC, 2011 to 2012, 17,379 hours. |
| **Target** | `cnt`, modelled as `log1p(cnt)` and scored back on the count scale. |
| **Excluded** | `casual` and `registered` (they sum to the target), `feels_like_c` for linear models (VIF over 60 with `temp_c`). |
| **Validation** | Train before 2012-10-01, test on Oct to Dec 2012. Tuning with expanding-window `TimeSeriesSplit`, so no fold sees the future. |
| **Chosen model** | LightGBM regressor. |

In [2]:
print("train:", reg["train_period"], f"({reg['n_train']:,} hours)")
print("test: ", reg["test_period"], f"({reg['n_test']:,} hours)")
pd.DataFrame(reg["test_metrics"]).T[["rmse", "mae", "r2"]]

train: ['2011-01-01 00:00:00', '2012-09-30 23:00:00'] (15,211 hours)
test:  ['2012-10-01 00:00:00', '2012-12-31 23:00:00'] (2,168 hours)


,rmse,mae,r2
baseline (hour x workingday mean),119.589,79.950,0.648
ridge,70.084,43.405,0.879
lasso,70.384,44.242,0.878
lightgbm,66.660,42.200,0.891


What works: the tree model and the regularised linear models all beat the hour by working-day baseline by a wide margin. The daily shape (commute peaks vs weekend hump) is captured well.

What does not: errors are largest in rain or snow, which is rare in training. Every test month is under-predicted on average because demand kept growing and the model only knows about growth through a year flag.

Limitations: it uses the actual weather of each hour. A real forecast would feed it forecast weather and be less accurate. Two years of data means it has seen each calendar month only twice.

## Model 2: term deposit subscription (classification)

| | |
|---|---|
| **Purpose** | Rank clients by how likely they are to subscribe, to decide who to call. |
| **Data** | UCI Bank Marketing (`bank-additional-full`), Portuguese bank, May 2008 to Nov 2010, 41,188 calls. |
| **Target** | `y` (subscribed), 11.27% positive. |
| **Excluded** | `duration`. Call length is only known after the call, when the outcome is already known, so it leaks the answer. `pdays` replaced by a `previously_contacted` flag. |
| **Validation** | Time ordered. First 80% of calls (May 2008 to May 2009) train, last 20% (May 2009 to Nov 2010) test. `TimeSeriesSplit` inside the training period. A random split is kept only as a reference. |
| **Chosen model** | Logistic regression without the macro columns, picked by time-series CV. Not fit for a call/no-call decision yet (see below). |

In [3]:
print(f"train {clf['n_train']:,} calls {clf['train_months']}, positive rate {clf['positive_rate_train']:.4f}")
print(f"test  {clf['n_test']:,} calls {clf['test_months']}, positive rate {clf['positive_rate_test']:.4f}")
print("\ntime-series CV inside the training period:")
display(pd.DataFrame(clf["cv_time_series"]).T)
print("test set (later calls):")
display(pd.DataFrame(clf["test_time_split"]).T)
print("random split vs time split ROC-AUC:")
pd.DataFrame(clf["random_vs_time_roc_auc"]).T

train 32,950 calls ['2008-05', '2009-05'], positive rate 0.0637
test  8,238 calls ['2009-05', '2010-11'], positive rate 0.3083

time-series CV inside the training period:


,roc_auc,roc_auc_min,pr_auc,brier
logistic,0.513,0.416,0.075,0.213
"logistic, no macro",0.547,0.430,0.103,0.067
lightgbm,0.494,0.437,0.072,0.070
"lightgbm, no macro",0.501,0.440,0.075,0.069


test set (later calls):


,roc_auc,pr_auc,within_month_auc,brier,mean_predicted,actual_rate
logistic,0.748,0.527,0.629,0.191,0.256,0.308
"logistic, no macro",0.701,0.496,0.635,0.223,0.175,0.308
lightgbm,0.658,0.445,0.618,0.250,0.093,0.308
"lightgbm, no macro",0.681,0.468,0.615,0.223,0.164,0.308


random split vs time split ROC-AUC:


,random split ROC-AUC,time split ROC-AUC,drop
logistic,0.801,0.748,0.052
"logistic, no macro",0.765,0.701,0.064
lightgbm,0.814,0.658,0.155
"lightgbm, no macro",0.782,0.681,0.101


In [4]:
print(f"decision model: {clf['decision_model']}, assumed cost of a wasted call = {clf['cost_fp']}, of a missed subscriber = {clf['cost_fn']}")
print(f"threshold chosen on the last block of the training period: {clf['threshold']}")
display(pd.DataFrame(clf["policies"]).set_index("policy"))
print("cost sensitivity:")
pd.DataFrame(clf["cost_sensitivity"])

decision model: logistic, no macro, assumed cost of a wasted call = 5, of a missed subscriber = 50
threshold chosen on the last block of the training period: 0.06


,calls,subscribers reached,missed,precision,recall,total cost
policy,,,,,,
call nobody,0,0,2540,NaN,0.000,127000.0
default threshold 0.5,889,462,2078,0.520,0.182,106035.0
cost-based threshold 0.06,5460,2071,469,0.379,0.815,40395.0
call everyone,8238,2540,0,0.308,1.000,28490.0


cost sensitivity:


,cost_fp,cost_fn,threshold,calls,cost_model,cost_default_0.5,cost_call_everyone,saving_vs_call_everyone
0,5,25,0.08,3859,30175.0,54085.0,28490.0,-0.059
1,5,50,0.06,5460,40395.0,106035.0,28490.0,-0.418
2,5,100,0.01,8238,28490.0,209935.0,28490.0,0.000
3,2,50,0.01,8238,11396.0,104754.0,11396.0,0.000
4,10,50,0.08,3859,60350.0,108170.0,56980.0,-0.059


In [5]:
print("top features by permutation importance on the later calls:", ", ".join(clf["top_permutation_features"]))

top features by permutation importance on the later calls: previously_contacted, month, job, contact, poutcome


What works: the time-ordered evaluation does what it should, which is stop me from shipping a model on the strength of a random-split score. Logistic regression ranks later calls at ROC-AUC 0.70 to 0.75 depending on the feature set, and having been contacted in an earlier campaign is the strongest client-level signal.

What does not: inside a month, where the bank actually chooses who to call, ranking is weak (about 0.62 to 0.64 for every model). Calibration breaks because conversion went from 6.37% in the training period to 30.83% in the test period. At my assumed costs no threshold picked from earlier data beats calling everyone, and every cost pair I tried says the same.

Limitations:

- The costs (5 per wasted call, 50 per missed subscriber) are my assumption, not the bank's. The sensitivity table shows the conclusion does not depend on them much.
- One time split is one draw. The test period is small months with a very different call list, so I would not read 0.748 vs 0.701 as a real difference between the two logistic models.
- The macro columns are close to a month label. They help a random split and hurt LightGBM on a time split, which is why I would keep them out or replace them with something that has a story, like the change in rates since the last campaign.
- Observational data from one bank in one period. Nothing here says calling someone on a mobile phone causes a subscription.
- `age`, `marital` and `job` are features. Before real use I would check how errors fall across age groups.

## Statistics, in one paragraph each

Sampling (notebook 2). On a skewed real population the sample mean was close to normal by n = 30, and both t and bootstrap intervals came out near 95% coverage.

Testing (notebook 3). The plain Welch test found no working-day difference in total daily rentals (p = 0.11), but once season and year were held fixed with HAC errors there was a borderline bump of about 191 rides a day (p = 0.048), which mostly goes away once weather is added too. Casual riders, contact type and weather all showed clear effects. With 41,188 rows nearly every categorical feature was "significant", so effect sizes did more of the filtering than p-values did. The simulated A/B test hit its planned false positive rate and power, and peeking 10 times nearly quadrupled the false positive rate.

## What I would do next

1. Bank: a rolling monthly evaluation (train on everything before month t, score month t), recalibrating on the latest labelled month each time.
2. Bank: an uplift model if there were treatment and control groups, since the real question is who changes their mind because of the call.
3. Bike: add a trend term and lagged demand for a day-ahead forecast, and test with forecast weather instead of observed weather.